# S4 - Error diagnostics and representation challenge

This stage compares word and character TF-IDF representations with
balanced and square-root balanced weights. No challenger passed all
three gates, so the baseline remains a diagnostic focus only. The tables
show the precision-recall trade-off for the critical class. Test, stress,
and monitor remain sealed. Broad char or transformer searches and
intensive tuning should use cloud or more RAM; this local run reached
its practical limit.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'dataset' / 'processed' / 'complaints.parquet').exists():
    parent = PROJECT_ROOT.parent
    if parent == PROJECT_ROOT:
        raise FileNotFoundError('Could not find project root')
    PROJECT_ROOT = parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from consumer_complaint_intelligence.config import ProjectPaths
from consumer_complaint_intelligence.s4 import run_s4
from consumer_complaint_intelligence.s4 import run_s4_smoke
from consumer_complaint_intelligence.s4_reporting import load_s4_report_tables

paths = ProjectPaths.from_root(PROJECT_ROOT)
RUN_MODE = 'disabled'
if RUN_MODE not in {'disabled', 'smoke', 'full'}:
    raise ValueError("RUN_MODE must be 'disabled', 'smoke', or 'full'")

scientific_cache = paths.temp_dir / 's3' / 'scientific.parquet'
artifact = paths.temp_dir / 's4' / 's4_results.json'
config = PROJECT_ROOT / 'config' / 's4_experiment.json'

if RUN_MODE == 'disabled':
    print('S4 disabled; no corpus or model was executed.')
    if artifact.exists():
        evidence = load_s4_report_tables(artifact)
        display(evidence.candidate_summary)
        display(evidence.critical_confusions)
        display(evidence.per_class)
elif RUN_MODE == 'smoke':
    run_s4_smoke(scientific_cache, artifact, config)
else:
    run_s4(scientific_cache, artifact, config)

if RUN_MODE != 'disabled':
    evidence = load_s4_report_tables(artifact)
    display(evidence.candidate_summary)
    display(evidence.critical_confusions)
    display(evidence.per_class)


S4 disabled; no corpus or model was executed.


candidate,representation,class_weight,macro_f1,weighted_f1,balanced_accuracy,critical_f1,critical_precision,critical_recall,eligible,recommended,diagnostic_focus
str,str,str,f64,f64,f64,f64,f64,f64,bool,bool,bool
"""word_balanced_reference""","""word""","""balanced""",0.700393,0.857715,0.741993,0.24581,0.173873,0.419283,false,false,true
"""word_sqrt_balanced""","""word""","""sqrt_balanced""",0.670119,0.859674,0.647878,0.0,0.0,0.0,false,false,false
"""char_wb_balanced""","""char_wb""","""balanced""",0.699701,0.857331,0.750801,0.233844,0.159754,0.436099,false,false,false
"""char_wb_sqrt_balanced""","""char_wb""","""sqrt_balanced""",0.697373,0.864964,0.680666,0.153846,0.639344,0.087444,false,false,false


candidate,diagnostic,true_class,predicted_class,count,rate
str,str,str,str,i64,f64
"""word_balanced_reference""","""critical_false_negative""","""debt_credit_management""","""debt_collection""",186,0.20852
"""word_balanced_reference""","""critical_false_negative""","""debt_credit_management""","""credit_reporting""",143,0.160314
"""word_balanced_reference""","""critical_false_negative""","""debt_credit_management""","""cards_prepaid""",51,0.057175
"""word_balanced_reference""","""critical_false_negative""","""debt_credit_management""","""consumer_lending""",44,0.049327
"""word_balanced_reference""","""critical_false_negative""","""debt_credit_management""","""mortgage""",39,0.043722
…,…,…,…,…,…
"""word_balanced_reference""","""top_global_confusion""","""debt_credit_management""","""money_services""",13,0.000363
"""word_balanced_reference""","""top_global_confusion""","""deposit_accounts""","""student_loan""",9,0.000251
"""word_balanced_reference""","""top_global_confusion""","""student_loan""","""money_services""",9,0.000251


candidate,product_family,precision,recall,f1,support
str,str,f64,f64,f64,i64
"""word_balanced_reference""","""cards_prepaid""",0.717413,0.735679,0.726431,17282
"""word_balanced_reference""","""consumer_lending""",0.559582,0.690957,0.618369,6436
"""word_balanced_reference""","""credit_reporting""",0.941057,0.911724,0.926158,164801
"""word_balanced_reference""","""debt_collection""",0.693733,0.675064,0.684271,25565
"""word_balanced_reference""","""debt_credit_management""",0.173873,0.419283,0.24581,892
"""word_balanced_reference""","""deposit_accounts""",0.754761,0.813751,0.783147,15635
"""word_balanced_reference""","""money_services""",0.658164,0.67589,0.666909,5421
"""word_balanced_reference""","""mortgage""",0.788688,0.888853,0.83578,6181
"""word_balanced_reference""","""student_loan""",0.77205,0.866737,0.816658,3767
